In [8]:
import pandas as pd
import numpy as np
import spacy
# import scispacy
import spacy
import re
from spacy.pipeline import EntityRuler
nlp = spacy.load("en_ner_bionlp13cg_md")
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC


In [9]:
## Loading the dfs
df = pd.read_csv("everything.csv", index_col=0)
df_doc = pd.read_csv('specialists.csv')

In [30]:
# Categorize.  Set dictionaries
col = ['label', 'anat_symp_lem', 'symp_lem']
df = df[col]
df = df[(pd.notnull(df['anat_symp_lem'])) & (pd.notnull(df['symp_lem']))]
df['category_id'] = df['label'].factorize()[0]
category_id_df = df[['label', 'category_id']].drop_duplicates().sort_values('category_id')
category_to_id = dict(category_id_df.values)
id_to_category = dict(category_id_df[['category_id', 'label']].values)


,label,anat_symp_lem,symp_lem,category_id
19,Ophthalmologist,mucus lymph node eye sore gland ...,eye redness - result inflammation widen tiny...,2
71,Ophthalmologist,uvea eye vitreous eyeball eye ...,print overview eye floater open pop-up dialog ...,2
72,Ophthalmologist,eye retinal detachment vitreous eye...,print overview retinal detachment open pop-up ...,2
73,Ophthalmologist,optic nerve tissue-like scar epiretina...,print overview part eye open pop-up dialog box...,2
74,Ophthalmologist,optic nerve eye cornea iris arte...,print overview open-angle glaucoma open pop-up...,2
...,...,...,...,...
930,Ophthalmologist,eye eyeglasse clear,wear eyeglass child eye do n t seem clear,2
931,Ophthalmologist,eye black float,black float front eye,2
932,Ophthalmologist,nettle,insect bite swell like nettle,2
933,Ophthalmologist,eye painful mirror,smoky eye also painful mirror,2


In [39]:
# Train model
X_train, X_test, y_train, y_test = train_test_split(df['anat_symp_lem'], df['label'], random_state = 42)
count_vect = CountVectorizer()
X_train_counts = count_vect.fit_transform(X_train)
tfidf_transformer = TfidfTransformer()
X_train_tfidf = tfidf_transformer.fit_transform(X_train_counts)
clf = LogisticRegression().fit(X_train_tfidf, y_train)


ValueError: Found input variables with inconsistent numbers of samples: [733, 978]

In [175]:
# USER INPUT
text = ["my head hurts and eyes are watering. ache in temple area."]
# text = ['my eyes hurt around the upper area. i am experiencing fever and ache.']
location = 'Pampanga'

In [176]:
doc = ''.join(list(clf.predict(count_vect.transform(text))))
doc

ValueError: X has 2 features per sample; expecting 1450

In [36]:
# Results
df_res = df_doc[(df_doc['Specialty ']==doc) & (df_doc['Location'].str.contains(location))]
df_online = df_doc.replace(np.nan, '', regex=True)
df_online = df_online[(df_online['Specialty ']==doc) & (df_online['Online']!="")]            

In [23]:
# Print results
if len(df_res) == 0:
    print(f"""      Searched for: {doc} in {location}
      None found in the list
      
      Another option is consulting via telemedicine with the following specialists:
      """)
    for index,row in df_online.iterrows():
        print(f"""
            Doctor's Name: {row["Doctor"]}
            Link for Telemedicine Consult: {row["Online"]}
        """)
else:
    print(f"       Searched for: {doc} in {location}")
    for index,row in df_res.iterrows():
        print(f"""
            Doctor's Name: {row["Doctor"]}
            Clinic Address: {(row["Clinic"])}
            Contact Details For Appointment: {row["Contact"]}
        """)

       Searched for: Ear, Nose, Throat in Pampanga

            Doctor's Name: Dr. Jeffrey Pangilinan
            Clinic Address: MacArthur Highway, Mabalacat, 2010 Pampanga
            Contact Details For Appointment: 0932 429 2684
        

            Doctor's Name: Dr. Joman Q. Laxamana
            Clinic Address: Km.78 MacArthur Highway, Brgy, San Fernando, 2000 Pampanga
            Contact Details For Appointment: 0932 856 4258
        


In [40]:
df = pd.read_csv("everything.csv", index_col=0)
df_doc = pd.read_csv('specialists.csv')

In [41]:
df

,label,symptoms,anatomic,anatomic_sent,symptoms_clean,symp_lem,symp_label,neg,site,anat_symp,anat_symp_lem,anat_lem
0,"Ear, Nose, Throat","['ear pain', 'itching and irritation in and ar...","{'sore glands', 'otitis externa', 'pus-like', ...","{This is known as chronic otitis externa.', 'S...",'ear pain 'itching irritation around ear canal...,ear pain itching irritation around ear can...,pain irritation redness swelling pressure full...,n't attempt squeeze pimples boils ear lead inf...,www.nhsinform.scot,"{'sore glands', 'otitis externa', 'pus-like', ...",sore gland otitis externa pus-like ...,sore gland otitis externa pus-like ...
1,"Ear, Nose, Throat","['frequent sneezing', 'runny or blocked nose',...","{'mucus', 'chest', 'eyes', ""nose'"", 'earache',...","{['frequent sneezing', 'runny or blocked nose'...",'frequent sneezing 'runny blocked nose 'itchy ...,frequent sneeze runny block nose itchy r...,sneezing blocked itchy red itchy throat cough ...,NaN,www.nhsinform.scot,"{'mucus', 'chest', 'eyes', ""nose'"", 'earache',...",mucus chest eye nose earache ...,mucus chest eye nose earache ...
2,"Ear, Nose, Throat","['a sore throat', 'a blocked or runny nose', '...","{'taste', 'eyes', 'muscles', 'muscle', 'ear', ...","{['a sore throat', 'a blocked or runny nose', ...",sore throat blocked runny nose 'sneezing cough...,sore throat block runny nose sneeze cough ho...,blocked runny nose sneezing cough hoarse voice...,NaN,www.nhsinform.scot,"{'taste', 'eyes', 'muscles', 'muscle', 'ear', ...",taste eye muscle muscle ear e...,taste eye muscle muscle ear e...
3,"Ear, Nose, Throat","['vertigo – the sensation that you, or the env...","{'ear', 'Ménière'}",{Some people may experience several attacks ea...,'vertigo - sensation environment around moving...,vertigo - sensation environment around move ...,vertigo spinning tinnitus hearing loss difficu...,NaN,www.nhsinform.scot,"{'ear', 'Ménière'}vertigo spinning tinnitus he...",ear ménière vertigo spin tinnitus hear lo...,ear ménière
4,"Ear, Nose, Throat","['a mild headache', 'nausea (feeling sick)', '...","{'eye', 'brain', 'leg', 'skull', 'body', 'brui...","{However, seek medical assistance if your chil...",mild headache 'nausea feeling sick 'mild dizzi...,mild headache nausea feel sick mild dizzin...,mild headache nausea vision unconsciousness di...,injury clear fluid nose could cerebrospinal fl...,www.nhsinform.scot,"{'eye', 'brain', 'leg', 'skull', 'body', 'brui...",eye brain leg skull body brui...,eye brain leg skull body brui...
...,...,...,...,...,...,...,...,...,...,...,...,...
975,"Ear, Nose, Throat",Suddenly my face could not move to the right. ...,"{'face', 'saliva', 'eyes'}",{Suddenly my face could not move to the right....,suddenly face could not move right saliva drip...,suddenly face could not move right saliva drip...,NaN,saliva dripping eyes,complaints,"{'face', 'saliva', 'eyes'}",face saliva eye,face saliva eye
976,Neurologist,"My grandfather had a stroke, he can no longer ...",set(),set(),grandfather stroke no longer walk feet,grandfather stroke no long walk foot,stroke,walk,complaints,()stroke,stroke,set
977,Neurologist,Our cousin's head turned. The motor collided a...,set(),set(),cousin 's head turned motor collided lost cons...,cousin s head turn motor collide lose consci...,NaN,NaN,complaints,(),,set
978,Neurologist,I feel like I will fall to the right when I walk,set(),set(),feel like fall right walk,feel like fall right walk,fall,NaN,complaints,()fall,fall,set


In [42]:
# Categorize.  Set dictionaries
df = pd.read_csv("everything.csv", index_col=0)
df_doc = pd.read_csv('specialists.csv')
col = ['label', 'anat_symp_lem', 'symp_lem']
df = df[col]
df = df[(pd.notnull(df['anat_symp_lem'])) & (pd.notnull(df['symp_lem']))]
df['category_id'] = df['label'].factorize()[0]
category_id_df = df[['label', 'category_id']].drop_duplicates().sort_values('category_id')
category_to_id = dict(category_id_df.values)
id_to_category = dict(category_id_df[['category_id', 'label']].values)

,label,anat_symp_lem,symp_lem,category_id
0,"Ear, Nose, Throat",sore gland otitis externa pus-like ...,ear pain itching irritation around ear can...,0
439,"Ear, Nose, Throat",tonsillar pharyngeal fever,mononucleosis classically fever lymphadenopath...,0
438,"Ear, Nose, Throat",thyroid gland innervate organ sinus ...,otalgia ear pain divide two broad category pri...,0
437,"Ear, Nose, Throat",posterior pharynx retropharyngeal abscess...,retropharyngeal abscess uncommon potentially l...,0
436,"Ear, Nose, Throat",cricopharyngeus esophagus submucosal l...,zenker s diverticulum type diverticulum deve...,0
...,...,...,...,...
391,Ophthalmologist,eyelashe red eyelid swollen eyelid eye...,blepharitis medical term eyelid inflammation c...,2
392,Ophthalmologist,eye eye twitch muscle eye eyelid...,eye twitch involuntary spasm muscle eyelid ble...,2
606,Ophthalmologist,eye cornea globe permanently [ eye...,ectropion outward turn eyelid margin typically...,2
386,Ophthalmologist,eye cornea eye eyelid interstiti...,keratitis inflammation cornea eye keratitis si...,2


In [47]:
from sklearn.model_selection import train_test_split


In [95]:
X_train, X_test, y_train, y_test = train_test_split(df['anat_symp_lem'], df['label'], random_state = 42)
count_vect = CountVectorizer()
X_train_counts = count_vect.fit_transform(X_train)
tfidf_transformer = TfidfTransformer()
X_train_tfidf = tfidf_transformer.fit_transform(X_train_counts)
clf = LogisticRegression().fit(X_train_tfidf, y_train)


In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

In [96]:
# Let's fetch all the possible text data
df = pd.read_csv("everything.csv", index_col=0)
df_doc = pd.read_csv('specialists.csv')
col = ['label', 'anat_symp_lem', 'symp_lem']
df = df[col]
df = df[(pd.notnull(df['anat_symp_lem'])) & (pd.notnull(df['symp_lem']))]
df['category_id'] = df['label'].factorize()[0]
category_id_df = df[['label', 'category_id']].drop_duplicates().sort_values('category_id')
category_to_id = dict(category_id_df.values)
id_to_category = dict(category_id_df[['category_id', 'label']].values)



In [97]:
df["data"] = df["anat_symp_lem"]

In [98]:
vectorizer = CountVectorizer()
vectorizer.fit(df.data)
vectorizer.vocabulary_

{'sore': 1322,
 'gland': 596,
 'otitis': 1013,
 'externa': 478,
 'pus': 1167,
 'like': 800,
 'jaw': 745,
 'watery': 1637,
 'ear': 410,
 'throat': 1478,
 'hair': 633,
 'follicle': 541,
 'skin': 1301,
 'pain': 1022,
 'irritation': 742,
 'redness': 1183,
 'swelling': 1428,
 'pressure': 1140,
 'fullness': 571,
 'discharge': 370,
 'tenderness': 1458,
 'swollen': 1430,
 'hear': 643,
 'loss': 817,
 'itch': 743,
 'discomfort': 372,
 'stenosis': 1354,
 'mirror': 884,
 'mucus': 904,
 'chest': 226,
 'eye': 486,
 'nose': 967,
 'earache': 411,
 'grain': 607,
 'sneeze': 1315,
 'block': 125,
 'itchy': 744,
 'red': 1181,
 'cough': 314,
 'postnasal': 1125,
 'drip': 386,
 'anosmia': 41,
 'facial': 495,
 'headache': 642,
 'tiredness': 1495,
 'fatigue': 503,
 'shortness': 1281,
 'breath': 149,
 'wheezing': 1641,
 'fever': 513,
 'taste': 1444,
 'muscle': 908,
 'face': 494,
 'runny': 1218,
 'hoarse': 659,
 'voice': 1626,
 'high': 656,
 'temperature': 1452,
 'cold': 269,
 'ménière': 915,
 'vertigo': 1603,
 '

In [110]:
# Converting our first sample into a vector
v0 = vectorizer.transform(df.data).toarray()


In [151]:
X_train, X_test, y_train, y_test = train_test_split(df['anat_symp_lem'], df['label'], test_size = 0.3, random_state=42)


In [152]:
vectorizer = CountVectorizer()
tfidfzer = TfidfTransformer()
lr = LogisticRegression()


In [153]:
vectors = vectorizer.fit_transform(X_train)
tfidf = tfidfzer.fit_transform(vectors)

In [154]:
# clf = LogisticRegression().fit(X_train_tfidf, y_train)


In [155]:
lr.fit(tfidf,y_train)

LogisticRegression()

In [156]:
vectors_test = vectorizer.transform(X_test)

In [157]:
pred = lr.predict(vectors_test)

In [164]:
from sklearn import metrics

acc_score = metrics.accuracy_score(y_test, pred)
f1_score = metrics.f1_score(y_test, pred, average='macro')

print('Total accuracy classification score: {}'.format(acc_score))
print('Total F1 classification score: {}'.format(f1_score))

Total accuracy classification score: 0.8741496598639455
Total F1 classification score: 0.875514754396742


In [165]:
from sklearn.metrics import classification_report
classification_report(y_test, pred)


'                   precision    recall  f1-score   support\n\nEar, Nose, Throat       0.90      0.83      0.87       102\n      Neurologist       0.82      0.87      0.85       101\n  Ophthalmologist       0.90      0.92      0.91        91\n\n         accuracy                           0.87       294\n        macro avg       0.88      0.88      0.88       294\n     weighted avg       0.88      0.87      0.87       294\n'

In [168]:

from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, pred)

array([[85, 15,  2],
       [ 6, 88,  7],
       [ 3,  4, 84]], dtype=int64)

In [174]:
text = ["my head hurts and eyes are watering. ache in temple area."]
doc = ''.join(list(lr.predict(vectorizer.transform(text))))
doc

'Neurologist'

In [178]:
import pickle


In [179]:
Pkl_Filename = "CheckApp_LR_Model.picle"


In [180]:
with open(Pkl_Filename, 'wb') as file:  
    pickle.dump(lr, file)